In [1]:
# Cell 1 — Install
!pip install transformers datasets peft bitsandbytes accelerate trl --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 16.5 MB/s eta 0:00:00


In [2]:
# Cell 2 — Imports
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
)
from datasets import load_dataset
from peft import PrefixTuningConfig, get_peft_model

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")
    print(f"VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch  : 2.10.0+cu128
CUDA     : True
GPU      : Tesla T4
VRAM     : 15.6 GB


In [ ]:
# Cell 3 — Config
#
# NUM_PREFIX_TOKENS = 20  (same count as the prompt tuning notebook)
# But prefix tuning trains FAR more params because these 20 tokens
# get projected into every layer's key and value matrices.
#
# Param math:
#   Prompt Tuning  : 20 × 3072              = 61,440  params

MODEL_NAME        = "unsloth/Llama-3.2-3B-Instruct"
MAX_SEQ_LENGTH    = 512
NUM_PREFIX_TOKENS = 20

In [5]:
# Cell 4 — Load tokenizer and model in 4bit
bnb_config = BitsAndBytesConfig(
    load_in_4bit              = True,
    bnb_4bit_quant_type       = "nf4",
    bnb_4bit_use_double_quant = True,
    bnb_4bit_compute_dtype    = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config = bnb_config,
    device_map          = "auto",
)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded")
print(f"Total params : {total_params:,}")
print(f"Hidden size  : {model.config.hidden_size}")
print(f"Num layers   : {model.config.num_hidden_layers}")
print(f"Num KV heads : {model.config.num_key_value_heads}")

config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

Model loaded
Total params : 1,803,463,680
Hidden size  : 3072
Num layers   : 28
Num KV heads : 8


In [6]:
# Cell 5 — Wrap model with PEFT Prefix Tuning
#
# KEY DIFFERENCES from Prompt Tuning:
#   1. PrefixTuningConfig instead of PromptTuningConfig
#   2. No "init text" — prefix tuning uses an MLP reparametrization
#      (a small network that maps a learned embedding → full prefix)
#      This is controlled by prefix_projection=True (default)
#   3. The prefix is injected into K,V of EVERY attention layer,
#      not just the input embedding layer

peft_config = PrefixTuningConfig(
    task_type          = "CAUSAL_LM",
    num_virtual_tokens = NUM_PREFIX_TOKENS,
    prefix_projection  = True,        # Use MLP reparametrization (recommended)
    encoder_hidden_size = 512,         # MLP bottleneck dimension
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

# You should see significantly more trainable params than prompt tuning
# because prefixes are injected at every layer's K and V

trainable params: 29,962,752 || all params: 3,242,712,576 || trainable%: 0.9240


In [14]:
# Cell 6 — Load, stratified sample (200 per category), and format
from collections import Counter

raw_dataset = load_dataset(
    "bitext/Bitext-customer-support-llm-chatbot-training-dataset",
    split="train",
)

# Stratified sampling: 200 examples per category (11 categories × 200 = 2,200)
categories = raw_dataset.unique("category")
print(f"Categories ({len(categories)}): {categories}")

sampled_indices = []
for cat in categories:
    cat_indices = [i for i, c in enumerate(raw_dataset["category"]) if c == cat]
    sampled_indices.extend(cat_indices[:200])

raw_dataset = raw_dataset.select(sampled_indices)
print(f"Sampled dataset : {len(raw_dataset):,} examples")
print(f"Distribution    : {Counter(raw_dataset['category'])}")

def format_prompt(examples):
    texts = []
    for instruction, response in zip(examples["instruction"], examples["response"]):
        text = (
            f"<|begin_of_text|>"
            f"<|start_header_id|>user<|end_header_id|>\n\n"
            f"{instruction}"
            f"<|eot_id|>"
            f"<|start_header_id|>assistant<|end_header_id|>\n\n"
            f"{response}"
            f"<|eot_id|>"
        )
        texts.append(text)
    return {"text": texts}

dataset = raw_dataset.map(format_prompt, batched=True)
print(f"\nFormatted dataset : {len(dataset):,} examples")
print(f"Sample            : {dataset[0]['text'][:200]}...")

README.md: 0.00B [00:00, ?B/s]

medical_meadow_medqa.json:   0%|          | 0.00/10.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10178 [00:00<?, ? examples/s]

Full dataset    : 10,178 examples
Sampled dataset : 500 examples


Map:   0%|          | 0/500 [00:00<?, ? examples/s]


Formatted dataset : 500 examples
Sample            :
<|begin_of_text|><|start_header_id|>user<|end_header_id|>

Please answer with one of the option in the bracket
Q:A 35-year-old woman comes to your office with a variety of complaints. As part of her evaluation, she undergoes laboratory testing which reveals the presence of anti-centromere antibodies...


In [11]:
# Cell 7 — Tokenize with label masking (loss only on assistant response)
ASSISTANT_HEADER = "<|start_header_id|>assistant<|end_header_id|>\n\n"
assistant_header_ids = tokenizer.encode(ASSISTANT_HEADER, add_special_tokens=False)
header_len = len(assistant_header_ids)

def tokenize(examples):
    tokens = tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding="max_length",
    )
    labels = []
    for ids in tokens["input_ids"]:
        label = list(ids)

        # Find the assistant header and mask everything up to (and including) it
        found = False
        for i in range(len(label) - header_len + 1):
            if label[i : i + header_len] == assistant_header_ids:
                label[: i + header_len] = [-100] * (i + header_len)
                found = True
                break

        if not found:
            label = [-100] * len(label)

        # Mask padding tokens
        pad_id = tokenizer.pad_token_id
        for i in range(len(label)):
            if ids[i] == pad_id:
                label[i] = -100

        labels.append(label)

    tokens["labels"] = labels
    return tokens

tokenized_dataset = dataset.map(
    tokenize,
    batched=True,
    remove_columns=dataset.column_names,
    num_proc=2,
)

# Quick sanity check: how many tokens per example are actually trained on?
import numpy as np
sample_labels = tokenized_dataset[0]["labels"]
n_train_tokens = sum(1 for t in sample_labels if t != -100)
n_masked = sum(1 for t in sample_labels if t == -100)
print(f"Tokenized dataset : {len(tokenized_dataset):,} examples")
print(f"Columns           : {tokenized_dataset.column_names}")
print(f"Sample label check: {n_train_tokens} trainable tokens, {n_masked} masked (of {len(sample_labels)})")

Map (num_proc=2):   0%|          | 0/2200 [00:00<?, ? examples/s]

Tokenized dataset : 2,200 examples
Columns           : ['input_ids', 'attention_mask', 'labels']
Sample label check: 46 trainable tokens, 466 masked (of 512)


In [13]:
# Cell 8 — Train
#
# gradient_accumulation_steps=1 for speed and lower peak VRAM
# effective batch size = 2 × 1 = 2  →  1,100 steps/epoch  →  3,300 steps total
# learning_rate = 3e-4 (stable for prefix tuning's ~30M params)

training_args = TrainingArguments(
    output_dir                  = "outputs",
    num_train_epochs            = 1,
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 1,
    learning_rate               = 3e-4,
    warmup_steps                = 30,
    weight_decay                = 0.01,
    lr_scheduler_type           = "cosine",
    bf16                        = torch.cuda.is_bf16_supported(),
    fp16                        = not torch.cuda.is_bf16_supported(),
    logging_steps               = 50,
    seed                        = 42,
    report_to                   = "none",
)

trainer = Trainer(
    model         = model,
    args          = training_args,
    train_dataset = tokenized_dataset,
    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)

trainer.train()

Step,Training Loss
50,1.347699
100,1.099356
150,0.938428
200,0.920266
250,0.890496
300,0.886405
350,0.853175
400,0.848368
450,0.807768
500,0.788853


TrainOutput(global_step=1100, training_loss=0.8294263527610085, metrics={'train_runtime': 8181.9309, 'train_samples_per_second': 0.269, 'train_steps_per_second': 0.134, 'total_flos': 1.90502223740928e+16, 'train_loss': 0.8294263527610085, 'epoch': 1.0})

In [15]:
# Cell 9 — Inference after training
model.eval()

test_questions = [
    "I need to cancel my order, how can I do that?",
    "I want to change the delivery address for my purchase.",
    "I haven't received my refund yet, can you help?",
    "How do I track my package?",
]

print("=" * 60)
print("AFTER TRAINING — Prefix Tuned Model Responses")
print("=" * 60)

for question in test_questions:
    prompt = (
        f"<|begin_of_text|>"
        f"<|start_header_id|>user<|end_header_id|>\n\n"
        f"{question}"
        f"<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>\n\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens = 150,
            temperature    = 0.7,
            do_sample      = True,
            pad_token_id   = tokenizer.eos_token_id,
        )

    prompt_len = inputs["input_ids"].shape[1]
    response   = tokenizer.decode(outputs[0][prompt_len:], skip_special_tokens=True)

    print(f"\nQ: {question}")
    print(f"A: {response.strip()}")
    print("-" * 60)

AFTER TRAINING — Prefix Tuned Model Responses


/usr/local/lib/python3.12/dist-packages/peft/peft_model.py:2141: UserWarning: Position ids are not supported for parameter efficient tuning. Ignoring position ids.
  warnings.warn("Position ids are not supported for parameter efficient tuning. Ignoring position ids.")



Q: I need to cancel my order, how can I do that?
A: I understand your need to cancel your order. To cancel your order, please follow these steps:

1. Log in to your account on our website or mobile app.
2. Navigate to the "My Orders" or "Order History" section.
3. Locate the order you wish to cancel and click on it.
4. Look for the "Cancel Order" or "Change Order" option.
5. Follow any additional instructions or prompts to complete the cancellation process.

If you encounter any difficulties or have further questions, please don't hesitate to reach out. We're here to assist you every step of the way. Let's work together to resolve this and make your cancellation as smooth as possible. How else can I assist you today?
------------------------------------------------------------

Q: I want to change the delivery address for my purchase.
A: No problem! I can assist you with changing the delivery address for your purchase. To make the necessary changes, please provide me with the new addr